In [13]:
import os
import json
import pandas as pd
from tqdm import tqdm

from kg.query import query_kg, query_kg_endpoint, get_triples_from_response
import matplotlib.pyplot as plt
import networkx as nx

from utils.prompts import gen_qa_prompt
from utils.bedrock_functions import invoke_bedrock_endpoint, build_anthropic_request_body
from utils.subgraph_functions import plot_graph_with_simplified_labels, prune_triples, restore_full_triples
from utils.helper_functions import read_jsonl_file, save_as_jsonl

In [2]:
directory = "/home/ec2-user/preetam_experiments/outputs/test"
split = 'test'
df = pd.read_json(f"{directory}/cleaned_subgraph_df_{split}.json", orient="records")

In [3]:
df

,subgraph_Steiner,subgraph_Steiner_length,subgraph_Steiner_largest_connected,subgraph_Steiner_largest_connected_length,QID,unwanted_filter_flag,unwanted_percentage
0,[[http://yago-knowledge.org/resource/Old__u002...,30,[[http://yago-knowledge.org/resource/Camelidae...,15,97855,False,0.000000
1,[[http://yago-knowledge.org/resource/Barry_Ban...,13,[[http://yago-knowledge.org/resource/Barry_Ban...,11,39548,False,0.000000
2,[[http://yago-knowledge.org/resource/Joseph_Ma...,16,[[http://yago-knowledge.org/resource/Joseph_Ma...,14,118880,False,0.000000
3,[[http://yago-knowledge.org/resource/Joe_Keena...,17,[[http://yago-knowledge.org/resource/Joe_Keena...,13,31930,False,0.000000
4,[[http://yago-knowledge.org/resource/Standards...,20,[[http://yago-knowledge.org/resource/Mzla_Tech...,17,97616,False,0.000000
...,...,...,...,...,...,...,...
19995,[[http://yago-knowledge.org/resource/Building_...,44,[[http://yago-knowledge.org/resource/Building_...,33,86508,True,45.454545
19996,[[http://yago-knowledge.org/resource/Reggie_Mi...,33,[[http://yago-knowledge.org/resource/Reggie_Mi...,33,50516,True,45.454545
19997,[[http://yago-knowledge.org/resource/Emma_Thom...,42,[[http://yago-knowledge.org/resource/Emma_Thom...,33,120462,True,45.454545
19998,"[[http://yago-knowledge.org/resource/China, ht...",17,"[[http://yago-knowledge.org/resource/China, ht...",11,94732,True,45.454545


In [4]:
str(df.iloc[0]['QID']).zfill(11)

'00000097855'

In [5]:
llm_requests_list = []

for i in tqdm(range(len(df))):
    recordId = str(df.iloc[i]['QID']).zfill(11)
    readable_triples = prune_triples(df.iloc[i]['subgraph_Steiner_largest_connected'])
    prompt = gen_qa_prompt(readable_triples)
    
    request_json = build_anthropic_request_body(
            system_prompt="You are helpful assistant.",
            user_prompt=prompt,
            max_tokens=2048,
            temperature=0
        )
    
    request_entry = {}
    request_entry['recordId'] = recordId
    request_entry['modelInput'] = request_json
    
    llm_requests_list.append(request_entry)

  0%|          | 0/20000 [00:00<?, ?it/s]

100%|██████████| 20000/20000 [00:03<00:00, 5427.59it/s]


In [9]:
save_as_jsonl(llm_requests_list, f"/home/ec2-user/preetam_experiments/batch_processing_files/{split}/llm_requests_{split}.jsonl") 
#f"{directory}/llm_requests_{split}.jsonl"


In [9]:
res_file = read_jsonl_file(f"llm_requests_{split}.jsonl.out")

In [12]:
res_file[0]['modelOutput']

{'id': 'msg_bdrk_01V58UM7i7VXVVnFiiUH4fNS',
 'type': 'message',
 'role': 'assistant',
 'model': 'claude-3-5-sonnet-20241022',
 'content': [{'type': 'text',
   'text': '{\n    "valid_qa_pairs": true,\n    "number_of_qa_pairs": 3,\n    "qa_pairs": [\n        {\n            "question": "What family of animals includes both the guanaco and llama?",\n            "answer": "Camelidae",\n            "supporting_path": [\n                {\n                    "subject": "Guanaco",\n                    "predicate": "parentTaxon",\n                    "object": "Lama__u0028_genus_u0029_"\n                },\n                {\n                    "subject": "Lama__u0028_genus_u0029_",\n                    "predicate": "parentTaxon",\n                    "object": "Camelidae"\n                },\n                {\n                    "subject": "Llama",\n                    "predicate": "parentTaxon",\n                    "object": "Lama__u0028_genus_u0029_"\n                }\n            ]\n 

In [50]:
df.iloc[99]['subgraph_Steiner_largest_connected']

[['http://yago-knowledge.org/resource/Mazda_B_series',
  'http://schema.org/manufacturer',
  'http://yago-knowledge.org/resource/Mazda'],
 ['http://yago-knowledge.org/resource/Troller_T4',
  'http://schema.org/manufacturer',
  'http://yago-knowledge.org/resource/Troller_Veículos_Especiais'],
 ['http://yago-knowledge.org/resource/Ford_Ranger',
  'http://schema.org/manufacturer',
  'http://yago-knowledge.org/resource/Ford_Motor_Company'],
 ['http://yago-knowledge.org/resource/Ford_Transit',
  'http://schema.org/manufacturer',
  'http://yago-knowledge.org/resource/Ford_Motor_Company'],
 ['http://yago-knowledge.org/resource/Maurice_Jordan_Q3300959',
  'http://schema.org/worksFor',
  'http://yago-knowledge.org/resource/PSA_Group'],
 ['http://yago-knowledge.org/resource/Maurice_Jordan_Q3300959',
  'http://schema.org/worksFor',
  'http://yago-knowledge.org/resource/Peugeot'],
 ['http://yago-knowledge.org/resource/1979_World_Rally_Championship_For_Manufacturers_Q20202921',
  'http://yago-knowl

In [47]:

resp = json.loads(res_file[99]['modelOutput']['content'][0]['text'])
resp

{'valid_qa_pairs': True,
 'number_of_qa_pairs': 3,
 'qa_pairs': [{'question': 'Which car manufacturer, who owns a company that makes the T4, participated in the 1979 World Rally Championship?',
   'answer': 'Ford_Motor_Company',
   'supporting_path': [{'subject': 'Troller_T4',
     'predicate': 'manufacturer',
     'object': 'Troller_Veículos_Especiais'},
    {'subject': 'Troller_Veículos_Especiais',
     'predicate': 'ownedBy',
     'object': 'Ford_Motor_Company'},
    {'subject': '1979_World_Rally_Championship_For_Manufacturers_Q20202921',
     'predicate': 'participant',
     'object': 'Ford_Motor_Company'}]},
  {'question': 'Which company that Maurice Jordan worked for was also a participant in the 1979 World Rally Championship?',
   'answer': 'Peugeot',
   'supporting_path': [{'subject': 'Maurice_Jordan_Q3300959',
     'predicate': 'worksFor',
     'object': 'Peugeot'},
    {'subject': '1979_World_Rally_Championship_For_Manufacturers_Q20202921',
     'predicate': 'participant',
  

In [55]:
print(resp['qa_pairs'][0]['question'])
print(resp['qa_pairs'][0]['answer'])
resp['qa_pairs'][0]['supporting_path']
restore_full_triples_universal(resp['qa_pairs'][0]['supporting_path'], df.iloc[99]['subgraph_Steiner_largest_connected'])
graph = crea

Which car manufacturer, who owns a company that makes the T4, participated in the 1979 World Rally Championship?
Ford_Motor_Company


([['http://yago-knowledge.org/resource/Troller_T4',
   'http://schema.org/manufacturer',
   'http://yago-knowledge.org/resource/Troller_Veículos_Especiais'],
  ['http://yago-knowledge.org/resource/Troller_Veículos_Especiais',
   'http://yago-knowledge.org/resource/ownedBy',
   'http://yago-knowledge.org/resource/Ford_Motor_Company'],
  ['http://yago-knowledge.org/resource/1979_World_Rally_Championship_For_Manufacturers_Q20202921',
   'http://yago-knowledge.org/resource/participant',
   'http://yago-knowledge.org/resource/Ford_Motor_Company']],
 True)

In [18]:
llm_requests_list[0]['modelInput']

{'anthropic_version': 'bedrock-2023-05-31',
 'system': 'You are helpful assistant.',
 'messages': [{'role': 'user',
   'content': "You are an AI assistant tasked with generating question-answer pairs from knowledge graph triples. Your goal is to create natural, human-like questions and their corresponding answers based on the provided graph data.\n\nTask Overview:\nGenerate **multi-hop, complex Q&A pairs** where the questions appear simple and natural but require reasoning across multiple connected relationships within the graph to infer the answer.\n\nGuidelines for Generating Q&A Pairs:\n1. **Question Design**:\n- Questions should utilize multiple connected relationships in the graph, requiring multi-hop reasoning.\n- Avoid single-hop or trivial questions directly derived from a single triple.\n- The answer should be an entity or node in the graph.\n\n2. **Multi-Hop Reasoning**:\n- Use paths connecting entities indirectly through multiple relationships to infer answers.\n- Questions 

In [19]:

model_id = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"
response_data = invoke_bedrock_endpoint(llm_requests_list[0]['modelInput'], model_id)
print("Response from Claude:", json.dumps(response_data, indent=2))

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


Response from Claude: {
  "id": "msg_bdrk_0135t5PoReNfhHVQYXTz6SDQ",
  "type": "message",
  "role": "assistant",
  "model": "claude-3-5-sonnet-20241022",
  "content": [
    {
      "type": "text",
      "text": "{\n    \"valid_qa_pairs\": true,\n    \"number_of_qa_pairs\": 3,\n    \"qa_pairs\": [\n        {\n            \"question\": \"What family of animals includes both the alpaca and guanaco, despite them belonging to different genera?\",\n            \"answer\": \"Camelidae\",\n            \"supporting_path\": [\n                {\n                    \"subject\": \"Alpaca\",\n                    \"predicate\": \"parentTaxon\",\n                    \"object\": \"Vicugna\"\n                },\n                {\n                    \"subject\": \"Vicugna\",\n                    \"predicate\": \"parentTaxon\",\n                    \"object\": \"Camelidae\"\n                },\n                {\n                    \"subject\": \"Guanaco\",\n                    \"predicate\": \"paren

In [28]:
resp = json.loads(response_data['content'][0]['text'])
restore_full_triples_universal(resp['qa_pairs'][0]['supporting_path'], df.iloc[0]['subgraph_Steiner_largest_connected'])

([['http://yago-knowledge.org/resource/Alpaca',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Vicugna'],
  ['http://yago-knowledge.org/resource/Vicugna',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Camelidae'],
  ['http://yago-knowledge.org/resource/Guanaco',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_'],
  ['http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Camelidae']],
 True)

In [27]:
json.loads(response_data['content'][0]['text'])

{'valid_qa_pairs': True,
 'number_of_qa_pairs': 3,
 'qa_pairs': [{'question': 'What family of animals includes both the alpaca and guanaco, despite them belonging to different genera?',
   'answer': 'Camelidae',
   'supporting_path': [{'subject': 'Alpaca',
     'predicate': 'parentTaxon',
     'object': 'Vicugna'},
    {'subject': 'Vicugna', 'predicate': 'parentTaxon', 'object': 'Camelidae'},
    {'subject': 'Guanaco',
     'predicate': 'parentTaxon',
     'object': 'Lama__u0028_genus_u0029_'},
    {'subject': 'Lama__u0028_genus_u0029_',
     'predicate': 'parentTaxon',
     'object': 'Camelidae'}]},
  {'question': 'Which higher-level taxonomic group connects both the Chevrotain and the Pecora through their evolutionary relationships?',
   'answer': 'Ruminant',
   'supporting_path': [{'subject': 'Chevrotain',
     'predicate': 'parentTaxon',
     'object': 'Tragulina'},
    {'subject': 'Tragulina', 'predicate': 'parentTaxon', 'object': 'Ruminant'},
    {'subject': 'Pecora', 'predicate'

In [6]:
request_json = build_anthropic_request_body(
            system_prompt="You are helpful assistant.",
            user_prompt=prompt,
            max_tokens=1024,
            temperature=0
        )


model_id = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"

        # Invoke the endpoint.
response_data = invoke_bedrock_endpoint(request_json, model_id)
print("Response from Claude:", json.dumps(response_data, indent=2))

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


Response from Claude: {
  "id": "msg_bdrk_01LeoVUs4q5o9TxWbnHnRwbP",
  "type": "message",
  "role": "assistant",
  "model": "claude-3-5-sonnet-20241022",
  "content": [
    {
      "type": "text",
      "text": "{\n    \"valid_qa_pairs\": true,\n    \"number_of_qa_pairs\": 3,\n    \"qa_pairs\": [\n        {\n            \"question\": \"What family of animals includes both the alpaca and the guanaco, despite them belonging to different genera?\",\n            \"answer\": \"Camelidae\",\n            \"supporting_path\": [\n                {\n                    \"subject\": \"Alpaca\",\n                    \"predicate\": \"parentTaxon\",\n                    \"object\": \"Vicugna\"\n                },\n                {\n                    \"subject\": \"Vicugna\",\n                    \"predicate\": \"parentTaxon\",\n                    \"object\": \"Camelidae\"\n                },\n                {\n                    \"subject\": \"Guanaco\",\n                    \"predicate\": \"p

In [8]:
resp = json.loads(response_data['content'][0]['text'])

In [1]:
resp

NameError: name 'resp' is not defined

In [13]:
lenjson.dumps(request_json)

'{"anthropic_version": "bedrock-2023-05-31", "system": "You are helpful assistant.", "messages": [{"role": "user", "content": "\\nYou are an AI assistant tasked with generating question-answer pairs from knowledge graph triples. Your goal is to create natural, human-like questions and their corresponding answers based on the provided graph data.\\n\\nTask Overview:\\nGenerate **multi-hop, complex Q&A pairs** where the questions appear simple and natural but require reasoning across multiple connected relationships within the graph to infer the answer.\\n\\nGuidelines for Generating Q&A Pairs:\\n1. **Question Design**:\\n- Questions should utilize multiple connected relationships in the graph, requiring multi-hop reasoning.\\n- Avoid single-hop or trivial questions directly derived from a single triple.\\n- The answer should be an entity or node in the graph.\\n\\n2. **Multi-Hop Reasoning**:\\n- Use paths connecting entities indirectly through multiple relationships to infer answers.\\n

In [9]:
restore_full_triples_universal(resp['qa_pairs'][0]['supporting_path'], df.iloc[idx]['subgraph_Steiner_largest_connected'])

([['http://yago-knowledge.org/resource/Alpaca',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Vicugna'],
  ['http://yago-knowledge.org/resource/Vicugna',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Camelidae'],
  ['http://yago-knowledge.org/resource/Guanaco',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_'],
  ['http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Camelidae']],
 True)

In [25]:
df.iloc[idx]['subgraph_Steiner_largest_connected']

[['http://yago-knowledge.org/resource/Camelidae',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Even-toed_ungulate'],
 ['http://yago-knowledge.org/resource/Even-toed_ungulate',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Ungulate'],
 ['http://yago-knowledge.org/resource/Tragulina',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Ruminant'],
 ['http://yago-knowledge.org/resource/Chevrotain',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Even-toed_ungulate'],
 ['http://yago-knowledge.org/resource/Chevrotain',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Tragulina'],
 ['http://yago-knowledge.org/resource/Camel',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Camelidae'],
 ['http://yago-knowledge.org/resource/Guanaco',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_'],
 ['http://ya

In [26]:
resp['qa_pairs'][0]['supporting_path']

[{'subject': 'Vicuña', 'predicate': 'parentTaxon', 'object': 'Vicugna'},
 {'subject': 'Vicugna', 'predicate': 'parentTaxon', 'object': 'Camelidae'},
 {'subject': 'Guanaco',
  'predicate': 'parentTaxon',
  'object': 'Lama__u0028_genus_u0029_'},
 {'subject': 'Lama__u0028_genus_u0029_',
  'predicate': 'parentTaxon',
  'object': 'Camelidae'}]